# LpProblem

## Overview

LpProblem represents a linear program over direct `Vector` and `Matrix` entries in `Values`. It combines linear costs with linear equality and inequality constraints and solves them with `ActiveSetSolver`.

LPs use the sparse active-set path. The dense QP subproblem option applies only to quadratic programs.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/constrained/doc/LpProblem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam

## Problem Model

An `LpProblem` stores:

- linear costs added with `addCost(LpCost)` or `addCost(JacobianFactor)`,
- equality constraints added with `LinearConstraint::Equal(...)`,
- inequality constraints added with `LinearConstraint::LessEqual(...)` or `LinearConstraint::GreaterEqual(...)`.

The objective is linear in the vectorized values. Constraints use the same `LinearConstraint` class as QPs, so row ordering, signs, and matrix-value vectorization are consistent across LP and QP problems.

## Minimal C++ Example

```cpp
using namespace gtsam;

const Key x = Symbol('x', 0);

LpProblem problem;
problem.addCost(JacobianFactor(
    x, (Matrix(1, 1) << -1.0).finished(), Vector::Zero(1)));
problem.addConstraint(LinearConstraint::LessEqual(JacobianFactor(
    x, Matrix::Identity(1, 1), (Vector(1) << 1.0).finished())));
problem.addConstraint(LinearConstraint::GreaterEqual(JacobianFactor(
    x, Matrix::Identity(1, 1), (Vector(1) << 0.0).finished())));

Values initial;
initial.insert(x, (Vector(1) << 0.0).finished());
Values result = problem.optimize(initial);
```

This example maximizes `x` by minimizing `-x` subject to `0 <= x <= 1`.

## Solving an LP

The convenience methods delegate to `ActiveSetSolver`:

```cpp
Values result = problem.optimize(initial);
Values resultWithoutInitial = problem.optimize();
```

When no initial point is supplied, the solver builds a phase-I LP with one slack variable to find a vector-valued feasible point. If the phase-I slack remains positive beyond `phaseOneFeasibilityTolerance`, the LP is reported infeasible.

## Active-Set Behavior

For LPs, each active-set iteration projects the negative objective gradient onto the current active constraint surface. If that direction is blocked by an inactive inequality, the solver steps to the first blocking row and activates it. If there is no blocking inequality and the projected direction descends forever, the solver reports the LP as unbounded.

The solver checks caller-provided initial values for feasibility. Infeasible initial values throw `std::invalid_argument`; unbounded LPs throw `std::runtime_error`.

## Parameters and Warm Starts

Use `ActiveSetSolverParams` to tune tolerances and iteration limits:

```cpp
auto params = std::make_shared<ActiveSetSolverParams>();
params->feasibilityTolerance = 1e-8;
params->multiplierTolerance = 1e-8;

ActiveSetSolver solver(problem, params);
auto [values, state] = solver.optimizeWithState(initial);
```

`ActiveSetSolverState` stores optimized values, row-ordered equality and inequality multipliers, the flattened active inequality rows, convergence status, and iteration count. Passing a previous state back to `optimizeWithState` warm-starts the active set when the constraint row layout is unchanged.

## Practical Guidance

Use `LpProblem` for linear objectives with linear constraints. Use `QpProblem` when the objective has a quadratic Hessian term.

LP active-set performance depends strongly on the number of constraints that become active and on the sparsity of the constraint rows. Start with the default sparse active-set solver. Tighten tolerances only when the application needs stricter feasibility or multiplier checks.